# 52 Flan LoRA Pipeline (large/xl/xxl)

Use a single config switch (`FLAN_VARIANT`) to run one Flan model at a time without overwriting previous Flan runs.


In [ ]:
# Cell 0: Configure variant, paths, and run toggles.
import os
import re
import ast
import json
import textwrap
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora')
EXPRESS_ROOT = PROJECT_ROOT / 'express-emotion-recognition'
REGISTRY_PATH = PROJECT_ROOT / 'configs' / 'model_registry.yaml'
ENV_FILE = PROJECT_ROOT / 'env.txt'
ENV_PKL_FILE = PROJECT_ROOT / 'configs' / 'env.pkl'

FLAN_VARIANT = 'xxl'  # choose: 'large', 'xl', 'xxl'
GPU_ID = None        # used when USE_MULTI_GPU=False
USE_MULTI_GPU = True
MULTI_GPU_IDS = '0,1'

# Distributed training controls (used by torchrun launch later).
DISTRIBUTED_TRAIN = True
TRAIN_GPU_IDS = '0,1'
TRAIN_NPROC_PER_NODE = 2
TRAIN_MASTER_PORT = 29661

RUN_TRAIN = True
RUN_INFERENCE = True
RUN_EVAL = True

TRAIN_ROWS = 12000   # use None for full express.csv
MAX_SOURCE_LEN = 512
MAX_TARGET_LEN = 32
MAX_NEW_TOKENS = 64
TRAIN_BATCH_SIZE = 4
INFER_BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 1e-4
EPOCHS = 2.0

# Memory tuning for large models (especially flan-t5-xxl).
LOW_MEM_MODE = True

if DISTRIBUTED_TRAIN:
    os.environ['CUDA_VISIBLE_DEVICES'] = TRAIN_GPU_IDS
elif USE_MULTI_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = MULTI_GPU_IDS
elif GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)


if LOW_MEM_MODE:
    if FLAN_VARIANT in {'xl', 'xxl'}:
        TRAIN_BATCH_SIZE = min(TRAIN_BATCH_SIZE, 1)
        INFER_BATCH_SIZE = min(INFER_BATCH_SIZE, 1)
        GRAD_ACCUM = max(GRAD_ACCUM, 8)
        MAX_SOURCE_LEN = min(MAX_SOURCE_LEN, 384)
        MAX_NEW_TOKENS = min(MAX_NEW_TOKENS, 48)
    else:
        TRAIN_BATCH_SIZE = min(TRAIN_BATCH_SIZE, 2)
        INFER_BATCH_SIZE = min(INFER_BATCH_SIZE, 4)

METRICS_TABLE_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'all_models_metrics.csv'


# Best-checkpoint and early stopping controls.
LOAD_BEST_MODEL_AT_END = True
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0
EVAL_SPLIT_RATIO = 0.1
MIN_EVAL_SAMPLES = 200

print('LOW_MEM_MODE =', LOW_MEM_MODE)
print('Effective TRAIN_BATCH_SIZE =', TRAIN_BATCH_SIZE)
print('Effective INFER_BATCH_SIZE =', INFER_BATCH_SIZE)
print('Effective GRAD_ACCUM =', GRAD_ACCUM)
print('Effective MAX_SOURCE_LEN =', MAX_SOURCE_LEN)
print('Effective MAX_NEW_TOKENS =', MAX_NEW_TOKENS)

print('USE_MULTI_GPU =', USE_MULTI_GPU)
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES'))


print('DISTRIBUTED_TRAIN =', DISTRIBUTED_TRAIN)
print('TRAIN_GPU_IDS =', TRAIN_GPU_IDS if DISTRIBUTED_TRAIN else os.environ.get('CUDA_VISIBLE_DEVICES'))
print('TRAIN_NPROC_PER_NODE =', TRAIN_NPROC_PER_NODE if DISTRIBUTED_TRAIN else 1)


In [ ]:
# Cell 1: Load token, registry, helper functions, and selected Flan spec.
import yaml
import pickle
import pandas as pd


def load_hf_token(env_path: Path, pkl_path: Path) -> str:
    if pkl_path.exists():
        try:
            with pkl_path.open('rb') as f:
                payload = pickle.load(f)
            token = str(payload.get('HF_TOKEN', '')).strip()
            if token:
                return token
        except Exception as exc:
            print(f'Warning: failed to read {pkl_path}: {exc}')

    if not env_path.exists():
        raise FileNotFoundError(f'Env file not found: {env_path}')

    token = ''
    for raw in env_path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip('"').strip("'")
            break

    if not token:
        raise ValueError(f'HF_TOKEN not found or empty in {env_path} or {pkl_path}')
    return token


def run_cmd(cmd, cwd=None, env=None, check=True):
    printable = ' '.join(str(x) for x in cmd)
    print(f'$ {printable}')
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)

    rc = proc.wait()
    out = ''.join(lines)
    if check and rc != 0:
        tail = ''.join(lines[-100:])
        raise RuntimeError(
            f'Command failed ({rc}): {printable}\n'
            f'--- Last output lines ---\n{tail}'
        )
    return out


def parse_eval_metrics(stdout_text):
    metrics = {}
    for key in ['VRate', 'AccL', 'AccV', 'F1V', 'AccV-2']:
        m = re.search(rf'{re.escape(key)}\s*=\s*([0-9]*\.?[0-9]+)', stdout_text)
        if m:
            metrics[key] = float(m.group(1))
    return metrics


HF_TOKEN = load_hf_token(ENV_FILE, ENV_PKL_FILE)
os.environ['HF_TOKEN'] = HF_TOKEN
print('Loaded HF token using', ENV_PKL_FILE, 'with fallback to', ENV_FILE)

registry = yaml.safe_load(REGISTRY_PATH.read_text(encoding='utf-8'))['models']
reg_map = {m['id']: m for m in registry}

SCRATCH_MODEL_ROOT = Path('/scratch/rameyjm7/models')

FLAN_MAP = {
    'large': {
        'model_id': 'flan_t5_large',
        'repo_id': 'google/flan-t5-large',
        'local_dir': SCRATCH_MODEL_ROOT / 'google__flan-t5-large',
        'size_b': 0.78,
        'baseline_file': 'flan_t5_large.csv',
        'label': 'Flan-T5-large + LoRA',
    },
    'xl': {
        'model_id': 'flan_t5_xl',
        'repo_id': 'google/flan-t5-xl',
        'local_dir': SCRATCH_MODEL_ROOT / 'google__flan-t5-xl',
        'size_b': 3.0,
        'baseline_file': 'flan_t5_xl.csv',
        'label': 'Flan-T5-xl + LoRA',
    },
    'xxl': {
        'model_id': 'flan_t5_xxl',
        'repo_id': 'google/flan-t5-xxl',
        'local_dir': SCRATCH_MODEL_ROOT / 'google__flan-t5-xxl',
        'size_b': 11.0,
        'baseline_file': 'flan_t5_xxl.csv',
        'label': 'Flan-T5-xxl + LoRA',
    },
}

if FLAN_VARIANT not in FLAN_MAP:
    raise ValueError(f'FLAN_VARIANT must be one of {list(FLAN_MAP.keys())}')

SPEC = FLAN_MAP[FLAN_VARIANT]
MODEL_ID = SPEC['model_id']
MODEL_NAME_OR_PATH = str(SPEC['local_dir']) if SPEC['local_dir'].exists() else SPEC['repo_id']
MODEL_SIZE_B = float(SPEC['size_b'])

MODEL_CACHE_DIR = PROJECT_ROOT / 'outputs' / 'hf_cache' / MODEL_ID
RUN_DIR = PROJECT_ROOT / 'outputs' / 'lora_runs' / f'{MODEL_ID}_express_lora'
METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics' / MODEL_ID
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures' / MODEL_ID

TRAIN_PREP_PATH = METRICS_DIR / f'{MODEL_ID}_train_prep_stats.json'
EVAL_INPUT_PATH = METRICS_DIR / f'{MODEL_ID}_baseline_eval_input.csv'
RAW_OUTPUT_PATH = METRICS_DIR / f'{MODEL_ID}_lora_raw.csv'
CLEAN_OUTPUT_PATH = METRICS_DIR / f'{MODEL_ID}_lora_clean.csv'
FIGURE_PATH = FIGURES_DIR / f'figure2_with_lora_overlay_{MODEL_ID}.png'

for p in [MODEL_CACHE_DIR, RUN_DIR, METRICS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('FLAN_VARIANT =', FLAN_VARIANT)
print('MODEL_ID =', MODEL_ID)
print('MODEL_NAME_OR_PATH =', MODEL_NAME_OR_PATH)
print('Expected local model dir =', SPEC['local_dir'])
print('Using local model dir =', SPEC['local_dir'].exists())
print('RUN_DIR =', RUN_DIR)
print('METRICS_DIR =', METRICS_DIR)

# Safety guard: keep outputs model-specific so variants never overwrite each other.
expected_tag = f"{MODEL_ID}_"
for out_path in [EVAL_INPUT_PATH, RAW_OUTPUT_PATH, CLEAN_OUTPUT_PATH, TRAIN_PREP_PATH, FIGURE_PATH]:
    if expected_tag not in out_path.name and MODEL_ID not in str(out_path):
        raise RuntimeError(f'Output path does not look model-specific: {out_path}')




In [ ]:
# Cell 2: Build train/eval dataframes for the selected Flan variant.
train_source = EXPRESS_ROOT / 'data' / 'express.csv'
eval_source = EXPRESS_ROOT / 'results' / SPEC['baseline_file']

train_df = pd.read_csv(train_source)
if TRAIN_ROWS is not None:
    train_df = train_df.head(TRAIN_ROWS)
train_df = train_df[['segment', 'labels', 'number_of_labels']].copy()

eval_df = pd.read_csv(eval_source)
eval_df = eval_df[['segment', 'labels', 'number_of_labels']].copy()
eval_df.to_csv(EVAL_INPUT_PATH, index=False)

print('Train rows:', len(train_df), 'from', train_source)
print('Eval rows:', len(eval_df), 'from', eval_source)
print('Saved eval input:', EVAL_INPUT_PATH)

display(train_df.head(2))
display(eval_df.head(2))


In [ ]:
# Cell 3: Train Flan LoRA (seq2seq) for the selected variant (DDP via torchrun).
if RUN_TRAIN:
    train_script_path = RUN_DIR / 'train_flan_lora_worker.py'
    train_csv_path = EXPRESS_ROOT / 'data' / 'express.csv'

    worker_script = 'import os\nimport ast\nimport json\nimport argparse\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\nfrom transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, EarlyStoppingCallback\nfrom peft import LoraConfig, TaskType, get_peft_model\n\n\ndef parse_labels(raw):\n    if isinstance(raw, list):\n        return [str(x).strip().lower() for x in raw]\n    if not isinstance(raw, str):\n        return None\n    try:\n        val = ast.literal_eval(raw)\n        if isinstance(val, list):\n            return [str(x).strip().lower() for x in val]\n    except Exception:\n        return None\n    return None\n\n\ndef make_prompt(segment, num_labels):\n    return (\n        "You are an assistant that predicts masked emotion words in a Reddit post.\\n"\n        f"Predict exactly {num_labels} masked token(s).\\n"\n        f"Return ONLY a Python list of lowercase emotion words with length {num_labels}.\\n\\n"\n        "Text:\\n"\n        f"{segment}\\n\\n"\n        "Answer:"\n    )\n\n\nclass Seq2SeqListDataset(Dataset):\n    def __init__(self, rows):\n        self.rows = rows\n    def __len__(self):\n        return len(self.rows)\n    def __getitem__(self, idx):\n        return self.rows[idx]\n\n\ndef collate_builder(tokenizer, max_source_len, max_target_len):\n    def collate_fn(features):\n        inputs = [f[\'input_text\'] for f in features]\n        targets = [f[\'target_text\'] for f in features]\n        model_inputs = tokenizer(inputs, max_length=max_source_len, truncation=True, padding=True, return_tensors=\'pt\')\n        target_batch = tokenizer(text_target=targets, max_length=max_target_len, truncation=True, padding=True, return_tensors=\'pt\')\n        labels = target_batch[\'input_ids\']\n        labels[labels == tokenizer.pad_token_id] = -100\n        model_inputs[\'labels\'] = labels\n        return model_inputs\n    return collate_fn\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--model_name_or_path\', required=True)\n    parser.add_argument(\'--train_source\', required=True)\n    parser.add_argument(\'--run_dir\', required=True)\n    parser.add_argument(\'--cache_dir\', required=True)\n    parser.add_argument(\'--train_prep_path\', required=True)\n    parser.add_argument(\'--train_rows\', type=int, default=-1)\n    parser.add_argument(\'--max_source_len\', type=int, default=512)\n    parser.add_argument(\'--max_target_len\', type=int, default=32)\n    parser.add_argument(\'--learning_rate\', type=float, default=1e-4)\n    parser.add_argument(\'--epochs\', type=float, default=2.0)\n    parser.add_argument(\'--batch_size\', type=int, default=1)\n    parser.add_argument(\'--grad_accum\', type=int, default=8)\n    parser.add_argument(\'--model_id\', required=True)\n    parser.add_argument(\'--low_mem_mode\', type=int, default=1)\n    parser.add_argument(\'--early_stopping_patience\', type=int, default=2)\n    parser.add_argument(\'--early_stopping_threshold\', type=float, default=0.0)\n    parser.add_argument(\'--eval_split_ratio\', type=float, default=0.1)\n    parser.add_argument(\'--min_eval_samples\', type=int, default=200)\n    args = parser.parse_args()\n\n    hf_token = os.environ.get(\'HF_TOKEN\')\n    if not hf_token:\n        raise RuntimeError(\'HF_TOKEN is required in environment for training.\')\n\n    run_dir = Path(args.run_dir)\n    run_dir.mkdir(parents=True, exist_ok=True)\n\n    tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, token=hf_token, cache_dir=args.cache_dir, use_fast=True)\n    train_df = pd.read_csv(args.train_source)\n    if args.train_rows is not None and args.train_rows >= 0:\n        train_df = train_df.head(args.train_rows)\n    train_df = train_df[[\'segment\', \'labels\', \'number_of_labels\']].copy()\n\n    samples = []\n    skipped = 0\n    for row in train_df.itertuples(index=False):\n        labels = parse_labels(row.labels)\n        if labels is None:\n            skipped += 1\n            continue\n        samples.append({\'input_text\': make_prompt(str(row.segment), int(row.number_of_labels)), \'target_text\': str(labels)})\n\n    prep = {\'total_rows\': int(len(train_df)), \'kept_rows\': int(len(samples)), \'skipped_rows\': int(skipped)}\n    rank = int(os.environ.get(\'RANK\', \'0\'))\n    if rank == 0:\n        Path(args.train_prep_path).write_text(json.dumps(prep, indent=2), encoding=\'utf-8\')\n        print(\'Train prep:\', prep)\n\n    if not samples:\n        raise RuntimeError(\'No valid training examples were built.\')\n\n    eval_dataset = None\n    if len(samples) >= max(args.min_eval_samples, 2):\n        eval_count = max(1, int(len(samples) * args.eval_split_ratio))\n        eval_rows = samples[-eval_count:]\n        train_rows = samples[:-eval_count]\n        if len(train_rows) >= 1 and len(eval_rows) >= 1:\n            train_dataset = Seq2SeqListDataset(train_rows)\n            eval_dataset = Seq2SeqListDataset(eval_rows)\n        else:\n            train_dataset = Seq2SeqListDataset(samples)\n    else:\n        train_dataset = Seq2SeqListDataset(samples)\n\n    base_model = AutoModelForSeq2SeqLM.from_pretrained(args.model_name_or_path, token=hf_token, cache_dir=args.cache_dir)\n    if args.low_mem_mode:\n        base_model.config.use_cache = False\n        try:\n            base_model.gradient_checkpointing_enable()\n        except Exception:\n            pass\n\n    lora_cfg = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32, lora_dropout=0.05, bias=\'none\', target_modules=[\'q\', \'v\'])\n    model = get_peft_model(base_model, lora_cfg)\n    if rank == 0:\n        model.print_trainable_parameters()\n\n    use_cuda = torch.cuda.is_available()\n    use_bf16 = bool(use_cuda and torch.cuda.is_bf16_supported())\n    use_fp16 = bool(use_cuda and not use_bf16)\n\n    eval_enabled = eval_dataset is not None\n    trainer_args = TrainingArguments(\n        output_dir=str(run_dir / \'trainer_state\'),\n        per_device_train_batch_size=args.batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        learning_rate=args.learning_rate,\n        num_train_epochs=args.epochs,\n        logging_steps=20,\n        save_strategy=\'epoch\',\n        eval_strategy=\'epoch\' if eval_enabled else \'no\',\n        load_best_model_at_end=bool(eval_enabled),\n        metric_for_best_model=\'eval_loss\' if eval_enabled else None,\n        greater_is_better=False if eval_enabled else None,\n        remove_unused_columns=False,\n        fp16=use_fp16,\n        bf16=use_bf16,\n        ddp_find_unused_parameters=False,\n        report_to=[],\n    )\n\n    callbacks = []\n    if eval_enabled:\n        callbacks.append(EarlyStoppingCallback(early_stopping_patience=args.early_stopping_patience, early_stopping_threshold=args.early_stopping_threshold))\n\n    trainer = Trainer(\n        model=model,\n        args=trainer_args,\n        train_dataset=train_dataset,\n        eval_dataset=eval_dataset,\n        data_collator=collate_builder(tokenizer, args.max_source_len, args.max_target_len),\n        callbacks=callbacks,\n    )\n\n    train_out = trainer.train()\n    model.save_pretrained(str(run_dir))\n    tokenizer.save_pretrained(str(run_dir))\n\n    if rank == 0:\n        summary = {\n            \'train_runtime\': float(getattr(train_out, \'metrics\', {}).get(\'train_runtime\', 0.0)),\n            \'train_samples\': int(len(train_dataset)),\n            \'timestamp_utc\': datetime.utcnow().isoformat() + \'Z\',\n            \'model_id\': args.model_id,\n            \'model_name_or_path\': args.model_name_or_path,\n            \'world_size\': int(os.environ.get(\'WORLD_SIZE\', \'1\')),\n        }\n        (run_dir / \'training_summary.json\').write_text(json.dumps(summary, indent=2), encoding=\'utf-8\')\n        print(\'Saved LoRA adapter to:\', run_dir)\n\n\nif __name__ == \'__main__\':\n    main()\n'
    train_script_path.write_text(worker_script, encoding='utf-8')
    print('Wrote training worker script to:', train_script_path)

    train_rows_arg = -1 if TRAIN_ROWS is None else int(TRAIN_ROWS)
    common_args = [
        '--model_name_or_path', str(MODEL_NAME_OR_PATH),
        '--train_source', str(train_csv_path),
        '--run_dir', str(RUN_DIR),
        '--cache_dir', str(MODEL_CACHE_DIR),
        '--train_prep_path', str(TRAIN_PREP_PATH),
        '--train_rows', str(train_rows_arg),
        '--max_source_len', str(MAX_SOURCE_LEN),
        '--max_target_len', str(MAX_TARGET_LEN),
        '--learning_rate', str(LR),
        '--epochs', str(EPOCHS),
        '--batch_size', str(TRAIN_BATCH_SIZE),
        '--grad_accum', str(GRAD_ACCUM),
        '--model_id', str(MODEL_ID),
        '--low_mem_mode', '1' if LOW_MEM_MODE else '0',
        '--early_stopping_patience', str(EARLY_STOPPING_PATIENCE),
        '--early_stopping_threshold', str(EARLY_STOPPING_THRESHOLD),
        '--eval_split_ratio', str(EVAL_SPLIT_RATIO),
        '--min_eval_samples', str(MIN_EVAL_SAMPLES),
    ]

    env = {
        'HF_TOKEN': HF_TOKEN,
        'TOKENIZERS_PARALLELISM': 'false',
        'PYTORCH_ALLOC_CONF': 'expandable_segments:True',
    }

    if DISTRIBUTED_TRAIN:
        cmd = [
            'torchrun',
            '--standalone',
            f'--nproc_per_node={int(TRAIN_NPROC_PER_NODE)}',
            f'--master_port={int(TRAIN_MASTER_PORT)}',
            str(train_script_path),
            *common_args,
        ]
    else:
        cmd = ['python', str(train_script_path), *common_args]

    _ = run_cmd(cmd, cwd=PROJECT_ROOT, env=env, check=True)
else:
    print('Training skipped (RUN_TRAIN=False).')


In [ ]:
# Cell 4: Run Flan LoRA inference for selected variant and save raw outputs.
if RUN_INFERENCE:
    import torch
    from tqdm import tqdm
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    from peft import PeftModel

    eval_df = pd.read_csv(EVAL_INPUT_PATH)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
        use_fast=True,
    )
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
    )
    model = PeftModel.from_pretrained(base_model, str(RUN_DIR))

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.eval()

    def make_prompt(segment, num_labels):
        return textwrap.dedent(f"""
        You are an assistant that predicts masked emotion words in a Reddit post.
        Predict exactly {num_labels} masked token(s).
        Return ONLY a Python list of lowercase emotion words with length {num_labels}.

        Text:
        {segment}

        Answer:
        """).strip()

    outputs = [None] * len(eval_df)

    for start in tqdm(range(0, len(eval_df), INFER_BATCH_SIZE), desc=f'{MODEL_ID} inference'):
        chunk = eval_df.iloc[start:start + INFER_BATCH_SIZE]
        prompts = [make_prompt(str(r.segment), int(r.number_of_labels)) for r in chunk.itertuples(index=False)]

        enc = tokenizer(
            prompts,
            max_length=MAX_SOURCE_LEN,
            truncation=True,
            padding=True,
            return_tensors='pt',
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.inference_mode():
            gen = model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )

        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        for i, txt in enumerate(decoded):
            outputs[start + i] = txt.strip()

    raw_df = eval_df.copy()
    raw_df['output'] = outputs
    raw_df.to_csv(RAW_OUTPUT_PATH, index=False)

    print('Saved raw outputs:', RAW_OUTPUT_PATH)
    print('Rows:', len(raw_df))
else:
    print('Inference skipped (RUN_INFERENCE=False).')


In [ ]:
# Cell 5: Clean and evaluate with existing scripts.
lora_metrics = {}
if RUN_EVAL:
    _ = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-cleaning.py'),
            '--input', str(RAW_OUTPUT_PATH),
            '--output', str(CLEAN_OUTPUT_PATH),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    eval_stdout = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-evaluation.py'),
            '--result_path', str(CLEAN_OUTPUT_PATH),
            '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    lora_metrics = parse_eval_metrics(eval_stdout)
    print('LoRA metrics:', lora_metrics)
else:
    print('Evaluation skipped (RUN_EVAL=False).')


In [ ]:
from datetime import datetime, timezone
# Cell 6: Upsert this Flan variant into shared metrics table.
row = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'model_id': MODEL_ID,
    'family': 'flan-t5',
    'model_name_or_path': MODEL_NAME_OR_PATH,
    'size_b': MODEL_SIZE_B,
    'pipeline': 'hf_peft_seq2seq',
    'is_lora': True,
    'v_rate': lora_metrics.get('VRate'),
    'accl': lora_metrics.get('AccL'),
    'accv': lora_metrics.get('AccV'),
    'f1v': lora_metrics.get('F1V'),
    'accv2': lora_metrics.get('AccV-2'),
    'clean_output_path': str(CLEAN_OUTPUT_PATH),
    'run_dir': str(RUN_DIR),
}

if METRICS_TABLE_PATH.exists():
    metrics_df = pd.read_csv(METRICS_TABLE_PATH)
else:
    metrics_df = pd.DataFrame()

if not metrics_df.empty and {'model_id', 'is_lora'}.issubset(metrics_df.columns):
    metrics_df = metrics_df[~((metrics_df['model_id'] == MODEL_ID) & (metrics_df['is_lora'].astype(str).str.lower() == 'true'))]

metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
metrics_df.to_csv(METRICS_TABLE_PATH, index=False)

print('Updated metrics table:', METRICS_TABLE_PATH)
display(metrics_df.tail(12))


In [ ]:
# Cell 7: Plot Figure 2 baseline + all completed Flan LoRA variants.
import seaborn as sns
import matplotlib.pyplot as plt

lex_df = pd.read_csv(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv')
vec_map = {str(r['word']).lower(): tuple(int(v) for v in r.iloc[1:].tolist()) for _, r in lex_df.iterrows()}
zero_vec = tuple([0] * 10)

_parse_cache = {}
def parse_list_cached(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if s == 'INVALID OUTPUT':
        return None
    if s in _parse_cache:
        return _parse_cache[s]
    try:
        v = ast.literal_eval(s)
        parsed = [str(x).strip().lower() for x in v] if isinstance(v, list) else None
    except Exception:
        parsed = None
    _parse_cache[s] = parsed
    return parsed


def compute_accv(csv_path):
    df = pd.read_csv(csv_path, usecols=['labels', 'output_formatted', 'number_of_labels'])
    total_masks = int(df['number_of_labels'].sum())
    matches = 0
    for raw_labels, raw_preds in zip(df['labels'].tolist(), df['output_formatted'].tolist()):
        labels = parse_list_cached(raw_labels)
        preds = parse_list_cached(raw_preds)
        if labels is None or preds is None:
            continue
        for t, p in zip(labels, preds):
            matches += int(vec_map.get(t, zero_vec) == vec_map.get(p, zero_vec))
    return (matches / total_masks) if total_masks > 0 else float('nan')

baseline_models = [
    {'label': 'Flan-T5-large', 'family': 'Flan-T5', 'size_b': 0.78, 'file': 'flan_t5_large.csv'},
    {'label': 'Flan-T5-xl', 'family': 'Flan-T5', 'size_b': 3.0, 'file': 'flan_t5_xl.csv'},
    {'label': 'Flan-T5-xxl', 'family': 'Flan-T5', 'size_b': 11.0, 'file': 'flan_t5_xxl.csv'},
    {'label': 'Gpt-3.5-turbo', 'family': 'GPT', 'size_b': 175.0, 'file': 'gpt_35_turbo_0125.csv'},
    {'label': 'Gpt-4o', 'family': 'GPT', 'size_b': 1750.0, 'file': 'gpt_4o.csv'},
    {'label': 'Gemma-2-2B-it', 'family': 'Gemma2', 'size_b': 2.0, 'file': 'gemma2_2B_it.csv'},
    {'label': 'Gemma-2-9B-it', 'family': 'Gemma2', 'size_b': 9.0, 'file': 'gemma2_9B_it.csv'},
    {'label': 'Gemma-2-27B-it', 'family': 'Gemma2', 'size_b': 27.0, 'file': 'gemma2_27B_it.csv'},
    {'label': 'Llama3.1-8B-instruct', 'family': 'Llama3.1', 'size_b': 8.0, 'file': 'llama3.1_8B_instruct.csv'},
    {'label': 'Llama3.1-70B-instruct', 'family': 'Llama3.1', 'size_b': 70.0, 'file': 'llama3.1_70B_instruct.csv'},
    {'label': 'Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'longformer.csv'},
    {'label': 'Mental-Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'mental-longformer.csv'},
    {'label': 'Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'roberta-base.csv'},
    {'label': 'Mental-Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'mental-roberta-base.csv'},
]

rows = []
for m in baseline_models:
    p = EXPRESS_ROOT / 'results' / m['file']
    rows.append({**m, 'accv': compute_accv(p)})
fig_df = pd.DataFrame(rows)

flan_lora_rows = []
for variant, spec in FLAN_MAP.items():
    clean_path = PROJECT_ROOT / 'outputs' / 'metrics' / spec['model_id'] / f"{spec['model_id']}_lora_clean.csv"
    if clean_path.exists():
        flan_lora_rows.append({
            'variant': variant,
            'model_id': spec['model_id'],
            'label': spec['label'],
            'size_b': float(spec['size_b']),
            'accv': compute_accv(clean_path),
        })

flan_lora_df = pd.DataFrame(flan_lora_rows)

sns.set_theme(style='whitegrid')
family_colors = {
    'Flan-T5': '#4C72B0',
    'GPT': '#DD8452',
    'Gemma2': '#55A868',
    'Llama3.1': '#C44E52',
    'Longformer': '#8172B2',
    'Roberta': '#937860',
}
family_markers = {
    'Flan-T5': 'D',
    'GPT': 'P',
    'Gemma2': '^',
    'Llama3.1': 'X',
    'Longformer': 's',
    'Roberta': 'o',
}

plt.figure(figsize=(14, 6))
for fam in ['Flan-T5', 'GPT', 'Gemma2', 'Llama3.1', 'Longformer', 'Roberta']:
    sub = fig_df[fig_df['family'] == fam].sort_values('size_b')
    if sub.empty:
        continue
    plt.plot(
        sub['size_b'], sub['accv'],
        marker=family_markers[fam],
        color=family_colors[fam],
        linestyle='--',
        linewidth=1.25,
        markersize=8,
        label=fam,
    )

if not flan_lora_df.empty:
    flan_lora_df = flan_lora_df.sort_values('size_b')
    plt.plot(
        flan_lora_df['size_b'], flan_lora_df['accv'],
        color='black', linestyle='-', linewidth=1.5, marker='*', markersize=14,
        label='Flan-T5 + LoRA'
    )
    for r in flan_lora_df.itertuples(index=False):
        plt.text(r.size_b * 1.04, r.accv + 0.004, r.label, fontsize=9)

plt.xscale('log')
plt.xlabel('Model Size')
plt.ylabel('Accuracy (Vector)')
y_max = 0.40
if not flan_lora_df.empty:
    y_max = max(y_max, float(flan_lora_df['accv'].max()) + 0.03)
plt.ylim(0.08, y_max)
plt.xticks([0.1, 1, 10, 100, 1000], ['100M', '1B', '10B', '100B', '1000B'])
plt.title('Figure 2 Baseline + Flan LoRA Variants')
plt.legend(loc='lower left')
plt.tight_layout()

plt.savefig(FIGURE_PATH, dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure:', FIGURE_PATH)
print('Completed Flan LoRA variants:', flan_lora_df['model_id'].tolist() if not flan_lora_df.empty else [])
